# II. Khám phá dữ liệu

Phần này thực hiện phân tích chi tiết về bộ dữ liệu, bao gồm kiểm tra chất lượng dữ liệu, phân tích các biến số và phân loại, phát hiện outliers và missing values, cùng với việc khám phá mối quan hệ giữa các biến.

## 2.1 Import Libraries và Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = pd.read_csv('../dataset/track_data_final.csv')
print(f"Dataset loaded: {len(df):,} rows")

## 2.2 Tổng quan về Dataset (Dataset Overview)

### Thông tin cơ bản (Basic Information)

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Mỗi hàng đại diện cho một bài hát (track) trên Spotify.

### Tính toàn vẹn của dữ liệu (Data Integrity)

In [ ]:
duplicated_by_id = df.duplicated(subset=['track_id']).sum()
duplicated_all = df.duplicated().sum()
empty_rows = df.isnull().all(axis=1).sum()

print(f"Duplicated by track_id: {duplicated_by_id:,}")
print(f"Duplicated rows (all columns): {duplicated_all:,}")
print(f"Empty rows: {empty_rows:,}")

Không có dữ liệu trùng lặp, không có dòng trống. Tất cả các `track_id` là duy nhất nên không cần xóa bản ghi trùng lặp (duplicate).

### Danh sách các cột (Column Inventory)

In [ ]:
df.info()

Ý nghĩa các cột:

Thông tin bài hát (Track Information):
- `track_id`: Mã định danh duy nhất của bài hát
- `track_name`: Tên bài hát
- `track_number`: Vị trí trong album
- `track_popularity`: Độ phổ biến (0-100)
- `track_duration_ms`: Thời lượng (milliseconds)
- `explicit`: Có nội dung nhạy cảm không

Thông tin nghệ sĩ (Artist Information):
- `artist_name`: Tên nghệ sĩ
- `artist_popularity`: Độ phổ biến nghệ sĩ (0-100)
- `artist_followers`: Số người theo dõi
- `artist_genres`: Thể loại âm nhạc

Thông tin Album (Album Information):
- `album_id`: Mã định danh album
- `album_name`: Tên album
- `album_release_date`: Ngày phát hành
- `album_total_tracks`: Tổng số bài
- `album_type`: Loại (album/single/compilation)

Các cột quan trọng cho phân tích: `track_popularity`, `artist_popularity`, `artist_followers`, `artist_genres`, `album_release_date`, `track_duration_ms`.

Không có cột cần bỏ vì tất cả đều có giá trị phân tích.

### Các kiểu dữ liệu (Data Types)

In [ ]:
df.dtypes

Cần chuyển đổi một số cột:
- `artist_popularity` và `artist_followers`: float64 -> int64 (sau khi xử lý missing)
- `album_release_date`: object -> datetime
- `track_duration_ms`: tạo thêm cột `track_duration_min` để dễ đọc hơn

In [ ]:
df['artist_popularity'] = df['artist_popularity'].fillna(0).astype('int64')
df['artist_followers'] = df['artist_followers'].fillna(0).astype('int64')
df['album_release_date'] = pd.to_datetime(df['album_release_date'], errors='coerce')
df['year'] = df['album_release_date'].dt.year
df['track_duration_min'] = df['track_duration_ms'] / 60000.0

## 2.3 Phân tích các cột Numerical (Numerical Columns Analysis)

In [ ]:
numerical_cols = ['track_popularity', 'artist_popularity', 'artist_followers', 
                  'track_duration_min', 'album_total_tracks', 'year']

df[numerical_cols].describe()

### Phân phối & Xu hướng tập trung (Distribution & Central Tendency)

In [ ]:
key_cols = ['track_popularity', 'artist_popularity', 'artist_followers', 'track_duration_min']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, col in enumerate(key_cols):
    data = df[col].dropna()
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)
    
    axes[i].hist(data, bins=50, edgecolor='black', alpha=0.6, density=True, color='skyblue')
    data.plot(kind='kde', ax=axes[i], color='red', linewidth=2)
    axes[i].set_title(f'{col}\nSkewness={skew:.2f}, Kurtosis={kurt:.2f}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Density')
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, col in enumerate(key_cols):
    df.boxplot(column=col, ax=axes[i], grid=False)
    axes[i].set_title(f'{col}')
    axes[i].set_ylabel('Value')

plt.tight_layout()
plt.show()

### Phạm vi & Giá trị ngoại lai (Range & Outliers)

In [ ]:
for col in key_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = outliers / len(df) * 100
    
    print(f"{col}: {outliers:,} outliers ({pct:.2f}%)")

Outliers trong `artist_followers` là các giá trị cực đoan trong thực tế - các siêu sao với hàng triệu followers. Các outliers khác cũng là giá trị hợp lý, không phải lỗi dữ liệu (data errors).

### Chất lượng dữ liệu (Data Quality)

In [ ]:
for col in numerical_cols:
    missing_pct = df[col].isnull().sum() / len(df) * 100
    min_val = df[col].min()
    max_val = df[col].max()
    zero_count = (df[col] == 0).sum()
    
    print(f"{col}:")
    print(f"Missing: {missing_pct:.2f}%")
    print(f"Range: [{min_val:.2f}, {max_val:.2f}]")
    print(f"Zeros: {zero_count:,}")
    print()

Không có giá trị âm hoặc không thể xảy ra. Giá trị 0 trong popularity là hợp lý (bài hát chưa được phổ biến). Giá trị 0 trong `artist_followers` đã được xử lý từ các giá trị bị thiếu.

## 2.4 Phân tích các cột Categorical (Categorical Columns Analysis)

### Phân phối giá trị (Value Distribution)

In [ ]:
categorical_cols = ['explicit', 'album_type', 'artist_name', 'artist_genres']

for col in categorical_cols:
    unique_count = df[col].nunique()
    missing_pct = df[col].isnull().sum() / len(df) * 100
    print(f"{col}: {unique_count:,} unique values, {missing_pct:.2f}% missing")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['explicit'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Explicit Content Distribution')
axes[0].set_xlabel('Explicit')
axes[0].set_ylabel('Count')

df['album_type'].value_counts().plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Album Type Distribution')
axes[1].set_xlabel('Album Type')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
print("Top 10 artists:")
df['artist_name'].value_counts().head(10)

Tất cả các cột categorical đều có phân bố mất cân bằng: `explicit` (75:25), `album_type` (69:26:5), `artist_name` (top artist chiếm ~4%), `artist_genres` (652 tổ hợp). Phân bố này có thể gây bias trong các mô hình Machine Learning và cần lưu ý khi phân tích các nhóm thiểu số.

### Chất lượng dữ liệu (Data Quality)

In [ ]:
for col in categorical_cols:
    missing_pct = df[col].isnull().sum() / len(df) * 100
    print(f"{col}: {missing_pct:.2f}% missing")

Tỷ lệ dữ liệu bị thiếu rất thấp (< 0.1%). Không có sự không nhất quán trong các danh mục vì `explicit` và `album_type` là các giá trị boolean/enum.

## 2.5 Phân tích dữ liệu bị thiếu (Missing Data Analysis)

In [ ]:
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Percent', ascending=False)

missing_summary[missing_summary['Missing_Count'] > 0]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

if len(missing_pct) > 0:
    missing_pct.plot(kind='barh', ax=ax, color='coral', edgecolor='black')
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column')
    ax.grid(alpha=0.3, axis='x')
    
    for i, v in enumerate(missing_pct):
        ax.text(v + 0.05, i, f'{v:.2f}%', va='center')
else:
    ax.text(0.5, 0.5, 'No missing values found', ha='center', va='center', fontsize=14)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

Giá trị bị thiếu (Missing values) chỉ xuất hiện ở:
- `year` (2.29%): Do lỗi phân tích ngày tháng (parse date) từ `album_release_date`
- `artist_name`, `artist_genres` (0.05%): 4 bài hát thiếu thông tin nghệ sĩ

Chiến lược xử lý: loại bỏ những dòng có chứa `NaN` vì tỷ lệ rất thấp.

## 2.6 Mối quan hệ & Tương quan (Relationships & Correlations)

In [ ]:
corr_cols = ['track_popularity', 'artist_popularity', 'artist_followers', 
             'track_duration_min', 'album_total_tracks']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Key Numerical Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

Tương quan giữa `track_popularity` và `artist_followers` khá yếu (0.226), cho thấy số lượng followers không trực tiếp quyết định độ phổ biến của bài hát. Ngược lại, `artist_popularity` có tương quan mạnh hơn nhiều với `track_popularity` (0.454), chứng tỏ danh tiếng hiện tại của nghệ sĩ quan trọng hơn quy mô cộng đồng người hâm mộ.

In [ ]:
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print("Top 5 strongest correlations:")
for col1, col2, corr in corr_pairs[:5]:
    print(f"{col1} <-> {col2}: {corr:.3f}")

### Bảng chéo (Cross-tabulations)

In [ ]:
pd.crosstab(df['explicit'], df['album_type'], margins=True)

### Thống kê tóm tắt theo nhóm (Grouped Summary Statistics)

In [ ]:
df.groupby('album_type')[['track_popularity', 'artist_popularity', 'track_duration_min']].mean()

In [ ]:
df.groupby('explicit')[['track_popularity', 'artist_popularity', 'artist_followers']].mean()

## 2.7 Quan sát ban đầu & Insight (Initial Observations & Insights)

### Các quan sát chính (Key Observations)

1.  Cả `track_popularity` và `artist_popularity` đều có phân bố lệch trái, cho thấy phần lớn các bài hát (tracks) và nghệ sĩ (artists) có độ phổ biến cao (50-80). Có sự tương quan mạnh giữa độ phổ biến của nghệ sĩ và độ phổ biến của bài hát (0.454).

2. `Artist followers`: Phân bố rất lệch phải (skewness=1.93) với nhiều outliers. Có 11.38% bài hát thuộc về nghệ sĩ siêu nổi tiếng (> 75M followers). Tương quan mạnh nhất là giữa `artist_popularity` và `artist_followers` (0.637).

3. `Track duration`: Trung bình 3.5 phút, phân bố khá tập trung quanh giá trị này. Có 4.29% outliers là các bài hát dài bất thường (> 6 phút).

4. `Explicit content`: 25% bài hát có nội dung nhạy cảm (explicit). Các bài hát explicit có độ phổ biến cao hơn (57.5 so với 50.5) và thường thuộc về nghệ sĩ nổi tiếng hơn.

5. `Album type`: Phần lớn là album (68.6%), đĩa đơn (singles) và tuyển tập (compilations) ít hơn. Bài hát từ albums có độ phổ biến cao nhất (55.5), singles thấp nhất (46.1).

### Các vấn đề về chất lượng dữ liệu (Data Quality Issues)

1. 201 bài hát (2.29%) thiếu thông tin năm do lỗi phân tích ngày tháng. Chỉ 4 bài hát (0.05%) thiếu thông tin nghệ sĩ.

2. 521 bài hát có độ phổ biến (popularity) = 0, có thể là bài hát mới chưa được đánh giá hoặc chưa phát hành công khai.

3. Bộ dữ liệu thiên về nghệ sĩ nổi tiếng (Taylor Swift chiếm 330 bài hát, 3.76%). Có thể gây bias khi phân tích xu hướng chung của ngành công nghiệp.

### Các bước tiền xử lý cần thiết (Preprocessing Steps Needed)

1. Xử lý giá trị bị thiếu (Handle missing values): Có thể loại bỏ nếu có `NaN`.

2. Xử lý giá trị ngoại lai (Outlier treatment): Giữ nguyên outliers vì chúng là các giá trị cực đoan thực tế (genuine extreme values), không phải lỗi.

3. Mã hóa (Encoding): One-hot encode cho `explicit` và `album_type`.

### Các mẫu thú vị (Interesting Patterns)

1. Nghệ sĩ nổi tiếng có tương quan mạnh với độ phổ biến của bài hát (track popularity). Điều này cho thấy brand name của nghệ sĩ ảnh hưởng lớn đến thành công của bài hát.

2. Các bài hát có nội dung nhạy cảm (explicit) có độ phổ biến trung bình cao hơn 7 điểm (57.5 so với 50.5), có thể do đối tượng khán giả hoặc chiến lược tiếp thị khác biệt.

3. Bài hát từ albums có hiệu suất (performance) tốt hơn singles, có thể do albums thường được quảng bá (promote) kỹ hơn và có cộng đồng người hâm mộ ổn định hơn.

### Cảnh báo & Hạn chế (Red Flags & Limitations)

1. Bộ dữ liệu (Dataset) tập trung vào các nghệ sĩ hàng đầu (top artists) (Taylor Swift 3.76%, The Weeknd 1.72%).

2. Không có thể loại (genre) cho từng bài hát gây khó khăn cho việc phân tích thể loại (genre analysis).

3. Thiếu thông tin về tempo, key, energy, danceability - các đặc trưng (features) quan trọng để phân tích đặc điểm âm nhạc (music characteristics).